# Chemistry Companion — Quick Start & API Tutorial

This notebook demonstrates how to use the core APIs of **Chemistry Companion** for structural analysis, spectral prediction, and descriptor generation.

## 1. Loading Molecules

The `core.molecule_utils` package provides robust parsing for SMILES and other formats.

In [ ]:
from core.molecule_utils import load_molecule

# Load aspirin
mol_rec = load_molecule('CC(=O)OC1=CC=CC=C1C(=O)O')
print(f'Name: {mol_rec.name}')
print(f'Formula: {mol_rec.formula}')
print(f'Exact Mass: {mol_rec.exact_mass:.2f}')

## 2. Functional Group Detection

Detect complex semantic functional groups out-of-the-box (27 standard categories).

In [ ]:
from spectra.functional_group_detector import detect_functional_groups

fg_report = detect_functional_groups(mol_rec.rdkit_mol)
print('Detected Groups:')
for match in fg_report.matches:
    print(f'- {match.name} (Count: {match.count})')

## 3. Spectral Prediction (IR & NMR)

Predict IR bands, ¹H NMR, and ¹³C NMR spectra using heuristic rules.

In [ ]:
from spectra.ir_predictor import predict_ir
from spectra.proton_nmr import predict_proton_nmr
from spectra.carbon_nmr import predict_carbon_nmr

ir_pred = predict_ir(mol_rec.rdkit_mol)
print('IR Bands:\n' + '-'*30)
for band in ir_pred.bands:
    print(f'{band.label}: {band.lower_cm1}-{band.upper_cm1} cm⁻¹ ({band.intensity})')

print('\n\n¹H NMR Signals:\n' + '-'*30)
h_pred = predict_proton_nmr(mol_rec.rdkit_mol)
for signal in h_pred.signals:
    print(f'{signal.shift_ppm} ppm | {signal.multiplicity} | {signal.label}')

print('\n\n¹³C NMR Environments:\n' + '-'*30)
c_pred = predict_carbon_nmr(mol_rec.rdkit_mol)
for env in c_pred.environments:
    print(f'{env.shift_ppm} ppm | {env.label}')

## 4. Using the Full Pipeline

The `ChemistryPipeline` orchestrates parsing, descriptors, functional groups, and spectral predictions into a single structured `AnalysisResult`.

In [ ]:
from core.pipeline import ChemistryPipeline
from core.config import get_settings

pipeline = ChemistryPipeline(settings=get_settings())
result = pipeline.process_smiles('c1ccccc1') # Benzene

print(f'Descriptors computed: {len(result.descriptors.to_dict())}')
print(f'IR Prediction summary:\n{result.ir_prediction.summary_text}')

## 5. Docking Preparation

Prepare 3D structures and assign Gasteiger charges natively for AutoDock Vina.

In [ ]:
from core.docking_preparation import prepare_docking_structure

# Generates 3D coordinates, adds hydrogens at pH 7.4, and saves PDBQT
result = prepare_docking_structure('CC(=O)O', output_dir='data/', filename_prefix='acetic_acid')
print(f'Generated PDBQT: {result}')